# Week 5: Support Vector Machines

For this week I am staying with the same digital marketing conversion dataset from Week 2 and Week 4. The target is still `Conversion`, so the main goal here is just to see how a few SVM setups compare without redoing all the older notebook work.

In [ ]:
import os

import kagglehub
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC

In [ ]:
# Same Kaggle dataset used in the Week 2/4 notebook
cached_path = os.path.expanduser(
    "~/.cache/kagglehub/datasets/rabieelkharoua/"
    "predict-conversion-in-digital-marketing-dataset/versions/1"
)

if os.path.exists(cached_path):
    path = cached_path
else:
    path = kagglehub.dataset_download(
        "rabieelkharoua/predict-conversion-in-digital-marketing-dataset"
    )

df_m = pd.read_csv(os.path.join(path, "digital_marketing_campaign_dataset.csv"))
df_m.head()

The older notebook had a lot of extra analysis, but SVM only needs the clean modeling setup: pick the target, split the data, scale the numeric columns, and encode categories if any show up.

In [ ]:
target = "Conversion"

features = [
    "Age",
    "Income",
    "AdSpend",
    "ClickThroughRate",
    "WebsiteVisits",
    "PagesPerVisit",
    "TimeOnSite",
    "EmailOpens",
    "EmailClicks",
    "PreviousPurchases",
    "LoyaltyPoints",
]

data = df_m[[target] + features].dropna()

X = data[features]
y = data[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

numeric_cols = X.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X.select_dtypes(exclude=np.number).columns.tolist()

print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)

In [ ]:
# Keep preprocessing in one place so every SVM gets the same inputs.
preprocess_steps = [("num", StandardScaler(), numeric_cols)]

if categorical_cols:
    try:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

    preprocess_steps.append(("cat", encoder, categorical_cols))

preprocessor = ColumnTransformer(preprocess_steps)

Now I am testing three SVM styles: linear, RBF, and polynomial. The `C` values are the regularization check, where smaller `C` is stricter and larger `C` lets the model fit the training data more closely.

In [ ]:
svm_models = [
    {"Model": "Linear SVM", "kernel": "linear"},
    {"Model": "RBF Kernel SVM", "kernel": "rbf"},
    {"Model": "Polynomial Kernel SVM", "kernel": "poly", "degree": 3},
]

c_values = [0.1, 1, 10]
results = []

for model_info in svm_models:
    for c_value in c_values:
        svm = SVC(
            C=c_value,
            kernel=model_info["kernel"],
            degree=model_info.get("degree", 3),
            probability=True,
            random_state=42,
        )

        pipe = Pipeline([
            ("preprocessor", preprocessor),
            ("svm", svm),
        ])

        pipe.fit(X_train, y_train)

        y_pred = pipe.predict(X_test)
        y_proba = pipe.predict_proba(X_test)[:, 1]
        tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

        results.append({
            "Model": model_info["Model"],
            "Kernel": model_info["kernel"],
            "C": c_value,
            "Accuracy": accuracy_score(y_test, y_pred),
            "ROC AUC": roc_auc_score(y_test, y_proba),
            "Precision": precision_score(y_test, y_pred, zero_division=0),
            "Recall": recall_score(y_test, y_pred, zero_division=0),
            "F1": f1_score(y_test, y_pred, zero_division=0),
            "Confusion Matrix": [[tn, fp], [fn, tp]],
            "TN": tn,
            "FP": fp,
            "FN": fn,
            "TP": tp,
        })

svm_results_df = pd.DataFrame(results)
svm_results_df.sort_values("ROC AUC", ascending=False).round(4)

In [ ]:
best_svm = svm_results_df.sort_values("ROC AUC", ascending=False).iloc[0]

print("Best model by ROC AUC")
print(best_svm[["Model", "C", "Accuracy", "ROC AUC", "Precision", "Recall", "F1"]])
print("Confusion matrix:")
print(np.array(best_svm["Confusion Matrix"]))

After this runs, I would mainly compare ROC AUC and F1 instead of accuracy alone. Accuracy can look decent even when the model is not catching both conversion classes very well, so the confusion matrix is helpful for seeing what is actually happening.